# Generate DS1

This notebook creates the public synthetic stand-in dataset used by `reproduce_paper.ipynb`.

Output file: `synthetic_data/DS1.xlsx`.

## 1. Configuration

The workbook contains only the columns needed by the reproduction workflow: 14 chemistry variables, `IMS`, `Oven Recipe`, measured targets, and cached NaMo-like estimates.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_DIR = Path.cwd().resolve()
OUTPUT_DIR = REPO_DIR / "synthetic_data"
OUTPUT_DIR.mkdir(exist_ok=True)
DS1_FILE = OUTPUT_DIR / "DS1.xlsx"

SEED = 52
TOTAL_ROWS = 9000
rng = np.random.default_rng(SEED)


## 2. Synthetic Design

The generator uses broad, hand-set alloy-family and aging-recipe effects. This creates visible dependencies among chemistry, recipe, NaMo-like estimates, and tensile properties without copying statistical details from the original industrial dataset.

In [ ]:
IMS_OPTIONS = [606035, 606385, 600524, 600563, 608226, 608227, 608260]
IMS_PROBABILITIES = np.array([0.16, 0.14, 0.15, 0.13, 0.15, 0.14, 0.13])
RECIPE_OPTIONS = [
    "Oven Recipe 08",
    "Oven Recipe 09",
    "Oven Recipe 14",
    "Oven Recipe 16",
    "Oven Recipe 20",
    "Oven Recipe 21",
]

STRENGTH_CODE = {
    606035: 0.0,
    606385: 0.3,
    600524: 0.8,
    600563: 1.1,
    608226: 1.5,
    608227: 1.7,
    608260: 2.0,
}

RECIPE_HEAT = {
    "Oven Recipe 08": 0.6,
    "Oven Recipe 09": 0.2,
    "Oven Recipe 14": -0.6,
    "Oven Recipe 16": 0.5,
    "Oven Recipe 20": -0.8,
    "Oven Recipe 21": 0.1,
}

RECIPE_STRENGTH = {
    "Oven Recipe 08": 18,
    "Oven Recipe 09": 8,
    "Oven Recipe 14": -10,
    "Oven Recipe 16": 15,
    "Oven Recipe 20": -16,
    "Oven Recipe 21": 4,
}

RECIPE_WEIGHTS = {
    606035: np.array([0.10, 0.18, 0.22, 0.10, 0.28, 0.12]),
    606385: np.array([0.14, 0.18, 0.14, 0.14, 0.20, 0.20]),
    600524: np.array([0.20, 0.24, 0.08, 0.18, 0.18, 0.12]),
    600563: np.array([0.22, 0.16, 0.10, 0.20, 0.18, 0.14]),
    608226: np.array([0.16, 0.14, 0.10, 0.22, 0.20, 0.18]),
    608227: np.array([0.18, 0.12, 0.08, 0.24, 0.18, 0.20]),
    608260: np.array([0.12, 0.12, 0.10, 0.20, 0.22, 0.24]),
}

def clip(values, low, high):
    return np.clip(values, low, high)


## 3. Generate the Minimal Table

The NaMo-like columns are deliberately imperfect and biased compared with the generated measured targets. That keeps the synthetic results meaningful for comparing NaMo, XGBoost, hybrid, and baseline behavior without encoding original confidential statistics.

In [ ]:
ims = rng.choice(IMS_OPTIONS, size=TOTAL_ROWS, p=IMS_PROBABILITIES)
recipes = np.array([rng.choice(RECIPE_OPTIONS, p=RECIPE_WEIGHTS[int(value)]) for value in ims])
strength = pd.Series(ims).map(STRENGTH_CODE).to_numpy()
heat = pd.Series(recipes).map(RECIPE_HEAT).to_numpy()
batch = rng.normal(0, 1, TOTAL_ROWS)
process = rng.normal(0, 1, TOTAL_ROWS)

Mg = clip(0.34 + 0.17 * strength + 0.04 * batch + rng.normal(0, 0.028, TOTAL_ROWS), 0.30, 1.15)
Si = clip(0.40 + 0.14 * strength + 0.04 * heat + rng.normal(0, 0.030, TOTAL_ROWS), 0.30, 1.10)
Mn = clip(0.005 + 0.035 * np.maximum(strength - 0.4, 0) + rng.normal(0, 0.008, TOTAL_ROWS), 0.0, 0.20)
Cu = clip(0.004 + 0.020 * np.maximum(strength - 0.7, 0) + 0.010 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.004, TOTAL_ROWS), 0.0, 0.14)
Cr = clip(0.002 + 0.010 * np.maximum(strength - 0.8, 0) + rng.normal(0, 0.003, TOTAL_ROWS), 0.0, 0.05)
Fe = clip(0.08 + 0.02 * rng.random(TOTAL_ROWS) + 0.012 * np.maximum(strength - 0.2, 0) + rng.normal(0, 0.009, TOTAL_ROWS), 0.03, 0.25)
B = clip(0.0010 + 0.0010 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.00025, TOTAL_ROWS), 0.0002, 0.006)
Ca = clip(0.0006 + 0.0018 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.0003, TOTAL_ROWS), 0.0001, 0.008)
Ni = clip(0.0008 + 0.0016 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.00025, TOTAL_ROWS), 0.0, 0.008)
Ti = clip(0.004 + 0.010 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.002, TOTAL_ROWS), 0.0, 0.05)
V = clip(0.002 + 0.010 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.002, TOTAL_ROWS), 0.0, 0.05)
Zn = clip(0.008 + 0.012 * rng.random(TOTAL_ROWS) + 0.010 * np.maximum(strength - 1.0, 0) + rng.normal(0, 0.003, TOTAL_ROWS), 0.0, 0.08)
Zr = clip(0.001 + 0.006 * np.maximum(strength - 0.9, 0) + rng.normal(0, 0.0015, TOTAL_ROWS), 0.0, 0.03)
Al = clip(100 - (B + Ca + Cr + Cu + Fe + Mg + Mn + Ni + Si + Ti + V + Zn + Zr) - clip(rng.normal(0.14, 0.03, TOTAL_ROWS), 0.06, 0.24), 97.6, 99.4)

signal = (
    220
    + 40 * strength
    + 54 * (Mg - 0.35)
    + 38 * (Si - 0.40)
    + 55 * Cu
    + 28 * Cr
    + 20 * Mn
    + pd.Series(recipes).map(RECIPE_STRENGTH).to_numpy()
    + 6 * batch
    + 4 * process
)
Rp02 = clip(signal + rng.normal(0, 6, TOTAL_ROWS), 210, 380)
Rm = clip(Rp02 + 18 + 9 * np.maximum(heat, 0) + 30 * np.maximum(Si - 0.45, 0) + rng.normal(0, 4, TOTAL_ROWS), Rp02 + 12, 400)

namo_signal = (
    228
    + 26 * strength
    + 42 * (Mg - 0.35)
    + 28 * (Si - 0.40)
    + 22 * Cu
    + 11 * Mn
    + 10 * heat
    - 9 * np.maximum(strength - 1.0, 0)
    + 3 * batch
)
NaMo_Rp02 = clip(namo_signal + np.where(strength < 0.4, 9, 0) + rng.normal(0, 11, TOTAL_ROWS), 200, 360)
NaMo_Rm = clip(NaMo_Rp02 + 16 + 7 * np.maximum(heat, 0) + rng.normal(0, 8, TOTAL_ROWS), NaMo_Rp02 + 8, 395)

ds1 = pd.DataFrame({
    "IMS": ims,
    "Oven Recipe": recipes,
    "Al": np.round(Al, 4),
    "B": np.round(B, 5),
    "Ca": np.round(Ca, 5),
    "Cr": np.round(Cr, 5),
    "Cu": np.round(Cu, 5),
    "Fe": np.round(Fe, 5),
    "Mg": np.round(Mg, 5),
    "Mn": np.round(Mn, 5),
    "Ni": np.round(Ni, 5),
    "Si": np.round(Si, 5),
    "Ti": np.round(Ti, 5),
    "V": np.round(V, 5),
    "Zn": np.round(Zn, 5),
    "Zr": np.round(Zr, 5),
    "Rp02": np.round(Rp02, 1),
    "Rm": np.round(Rm, 1),
    "NaMo Rp02": np.round(NaMo_Rp02, 1),
    "NaMo Rm": np.round(NaMo_Rm, 1),
})
ds1.index = np.arange(100000, 100000 + len(ds1))


## 4. Save and Inspect

The reproduction notebook creates cross-validation folds internally from this single workbook.

In [ ]:
ds1.to_excel(DS1_FILE)

print(f"Saved DS1 workbook: {DS1_FILE}")
print(f"Shape: {ds1.shape}")
print("Columns:")
print(list(ds1.columns))

display(ds1.head())
display(ds1.groupby("IMS")[["Rp02", "Rm", "NaMo Rp02", "NaMo Rm"]].mean().round(2))
